'''
    @Author:Ankitha
    @Date: 03-01-2025
    @Last Modified by: Ankitha
    @Last Modified time: 03-01-2025
    @Title :PySpark Sql programs
'''

In [2]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName("covid_dataset").getOrCreate()

df1 = spark.read.csv("country_wise_latest.csv",header=True,inferSchema=True)
df2 = spark.read.csv("covid_19_clean_complete.csv",header=True,inferSchema=True)
df3 = spark.read.csv("day_wise.csv",header=True,inferSchema=True)
df4 = spark.read.csv("full_grouped.csv",header=True,inferSchema=True)
df5 = spark.read.csv("usa_county_wise.csv",header=True,inferSchema=True)
df6 = spark.read.csv("worldometer_data.csv",header=True,inferSchema=True)


In [6]:
df1.createOrReplaceTempView("df1")
df2.createOrReplaceTempView("df2")
df3.createOrReplaceTempView("df3")
df4.createOrReplaceTempView("df4")
df5.createOrReplaceTempView("df5")
df6.createOrReplaceTempView("df6")

### Q1.Total Conformed cases by date

In [13]:
total_confirmed_bydate = """ 
select Date,sum(Confirmed) as total_confirmed
from df3
group by Date
order by Date
"""
result = spark.sql(total_confirmed_bydate)
result.show(5)

+----------+---------------+
|      Date|total_confirmed|
+----------+---------------+
|2020-01-22|            555|
|2020-01-23|            654|
|2020-01-24|            941|
|2020-01-25|           1434|
|2020-01-26|           2118|
+----------+---------------+
only showing top 5 rows



### Q2.Total deaths and recovers over time

In [17]:
total_deaths_over_time = """ 
select Date,sum(Deaths) as Total_deaths,sum(Recovered) as Total_recovered
from df3
group by Date
"""
result = spark.sql(total_deaths_over_time)
result.show(5)

+----------+------------+---------------+
|      Date|Total_deaths|Total_recovered|
+----------+------------+---------------+
|2020-07-24|      639650|        8939705|
|2020-04-30|      234704|         989616|
|2020-03-07|        3553|          56760|
|2020-03-13|        5406|          68359|
|2020-02-04|         492|            821|
+----------+------------+---------------+
only showing top 5 rows



### Q3.Active Cases by Date

In [19]:
total_active_cases = """ 
select Date,sum(Active) as Totoal_activecases
from df3
group by Date
"""
result = spark.sql(total_active_cases)
result.show(5)

+----------+------------------+
|      Date|Totoal_activecases|
+----------+------------------+
|2020-07-24|           6212290|
|2020-04-30|           2044556|
|2020-03-07|             44999|
|2020-03-13|             72243|
|2020-02-04|             22585|
+----------+------------------+
only showing top 5 rows



### Q4.Summarize the Data by WHO Region

In [ ]:
# Corrected SQL query for column names with spaces
regional_summary_sql = """
SELECT `WHO Region`, 
       SUM(Confirmed) AS Total_Confirmed, 
       SUM(Deaths) AS Total_Deaths, 
       SUM(Recovered) AS Total_Recovered, 
       SUM(Active) AS Total_Active
FROM df1
GROUP BY `WHO Region`
ORDER BY Total_Confirmed DESC
"""

regional_summary_df = spark.sql(regional_summary_sql)

regional_summary_df.show()


+--------------------+---------------+------------+---------------+------------+
|          WHO Region|Total_Confirmed|Total_Deaths|Total_Recovered|Total_Active|
+--------------------+---------------+------------+---------------+------------+
|            Americas|        8839286|      342732|        4468616|     4027938|
|              Europe|        3299523|      211144|        1993723|     1094656|
|     South-East Asia|        1835297|       41349|        1156933|      637015|
|Eastern Mediterra...|        1490744|       38339|        1201400|      251005|
|              Africa|         723207|       12223|         440645|      270339|
|     Western Pacific|         292428|        8249|         206770|       77409|
+--------------------+---------------+------------+---------------+------------+



### Q5.Country with lowest Number of Deaths

In [34]:
regional_confirmed_cases = """
SELECT `Country/Region`, MIN(Deaths) AS Total_deaths
FROM df1
GROUP BY `Country/Region`
ORDER BY Total_deaths ASC
LIMIT 1
"""

result = spark.sql(regional_confirmed_cases)
result.show()


+--------------+------------+
|Country/Region|Total_deaths|
+--------------+------------+
|       Eritrea|           0|
+--------------+------------+



### Q6.Calculating 100% recovery rates

In [64]:
percent_recovery_rates = """ 
SELECT `Country/Region`,
       `Confirmed`,
       `Recovered`,
       `Deaths`,
       ROUND((`Recovered` / `Confirmed`) * 100, 2) AS `Recovery_Rate`
FROM df1
WHERE `Confirmed` > 0
  AND ROUND((`Recovered` / `Confirmed`) * 100, 2) = 100
ORDER BY `Recovery_Rate` DESC 
LIMIT 5
"""
result = spark.sql(percent_recovery_rates)
result.show()


+--------------+---------+---------+------+-------------+
|Country/Region|Confirmed|Recovered|Deaths|Recovery_Rate|
+--------------+---------+---------+------+-------------+
|      Dominica|       18|       18|     0|        100.0|
|       Grenada|       23|       23|     0|        100.0|
|      Holy See|       12|       12|     0|        100.0|
+--------------+---------+---------+------+-------------+



### Q7.Country with High Recovery Rates

In [66]:
percent_recovery_rates = """ 
SELECT `Country/Region`,
       `Confirmed`,
       `Recovered`,
       ROUND((`Recovered` / `Confirmed`) * 100, 2) AS `Recovery_Rate`
FROM df1
WHERE `Confirmed` > 0
  AND ROUND((`Recovered` / `Confirmed`) * 100, 2) > 89.99
ORDER BY `Recovery_Rate` DESC
LIMIT 5
"""

result = spark.sql(percent_recovery_rates)
result.show(truncate=False)


+--------------+---------+---------+-------------+
|Country/Region|Confirmed|Recovered|Recovery_Rate|
+--------------+---------+---------+-------------+
|Dominica      |18       |18       |100.0        |
|Holy See      |12       |12       |100.0        |
|Grenada       |23       |23       |100.0        |
|Djibouti      |5059     |4977     |98.38        |
|Iceland       |1854     |1823     |98.33        |
+--------------+---------+---------+-------------+



### Q8.Notable Nations with High Case Numbers

In [72]:
highest_cases = """ 
SELECT `Country/Region`,
       SUM(`Confirmed`) AS total_cases,
       SUM(`Recovered`) AS total_recovered_cases
FROM df1
GROUP BY `Country/Region`
order by total_cases limit 5
"""

result = spark.sql(highest_cases)
result.show()


+--------------------+-----------+---------------------+
|      Country/Region|total_cases|total_recovered_cases|
+--------------------+-----------+---------------------+
|      Western Sahara|         10|                    8|
|            Holy See|         12|                   12|
|           Greenland|         14|                   13|
|Saint Kitts and N...|         17|                   15|
|            Dominica|         18|                   18|
+--------------------+-----------+---------------------+



### Q9.comparison Of Recovery And Fatality Rates By Country

In [73]:
recovery_fatality_rates = """
SELECT `Country/Region`,
       `Confirmed`,
       `Recovered`,
       `Deaths`,
       ROUND((`Recovered` / `Confirmed`) * 100, 2) AS `Recovery_Rate`,
       ROUND((`Deaths` / `Confirmed`) * 100, 2) AS `Fatality_Rate`
FROM df1
"""
result = spark.sql(recovery_fatality_rates)
top_recovery_fatality = result.sort(
    ['Recovery_Rate', 'Fatality_Rate'], ascending=[False, True]
).limit(10)
top_recovery_fatality.show(5)


+--------------+---------+---------+------+-------------+-------------+
|Country/Region|Confirmed|Recovered|Deaths|Recovery_Rate|Fatality_Rate|
+--------------+---------+---------+------+-------------+-------------+
|       Grenada|       23|       23|     0|        100.0|          0.0|
|      Dominica|       18|       18|     0|        100.0|          0.0|
|      Holy See|       12|       12|     0|        100.0|          0.0|
|      Djibouti|     5059|     4977|    58|        98.38|         1.15|
|       Iceland|     1854|     1823|    10|        98.33|         0.54|
+--------------+---------+---------+------+-------------+-------------+
only showing top 5 rows



### Q10.Country with lowest number of deaths

In [ ]:
country_lowest_death = """
SELECT `Country/Region`, 
       MIN(`Deaths`) AS lowest_deaths
FROM df1
GROUP BY `Country/Region`
ORDER BY lowest_deaths ASC
"""

result = spark.sql(country_lowest_death)
result.show()


+--------------+-------------+
|Country/Region|lowest_deaths|
+--------------+-------------+
|      Cambodia|            0|
|   Timor-Leste|            0|
+--------------+-------------+
only showing top 2 rows



### Q11. Countries with the Highest Number of COVID-19 Cases

In [82]:
highest_num_cases = """ 
select `Country/Region`,
        max(Confirmed) as highest_case
from df1
group by `Country/Region`
order by highest_case desc
"""
result = spark.sql(highest_num_cases)
result.show(2)

+--------------+------------+
|Country/Region|highest_case|
+--------------+------------+
|            US|     4290259|
|        Brazil|     2442375|
+--------------+------------+
only showing top 2 rows



### Q12.Global Recovery Rate

In [83]:
recovery_rate = """ 
select `Country/Region`,Round((sum(Confirmed)/sum(Recovered))*100,2) as recovery_rate
from df1
group by `Country/Region`
"""
result = spark.sql(recovery_rate)
result.show()

+--------------+-------------+
|Country/Region|recovery_rate|
+--------------+-------------+
|          Chad|       113.83|
|      Paraguay|       156.56|
|        Russia|       135.61|
|         Yemen|        203.0|
|       Senegal|       150.75|
|    Cabo Verde|       150.19|
|        Sweden|         NULL|
|        Guyana|       214.92|
|         Burma|       119.86|
|       Eritrea|       138.74|
|   Philippines|       310.22|
|      Djibouti|       101.65|
|      Malaysia|       103.52|
|     Singapore|       111.26|
|          Fiji|        150.0|
|        Turkey|       107.86|
|        Malawi|       222.74|
|Western Sahara|        125.0|
|          Iraq|       145.94|
|       Germany|       108.83|
+--------------+-------------+
only showing top 20 rows



### Q13.COVID-19 Trends By Continent

In [86]:
trend_by_continent = """ 
select Continent,sum(TotalCases) as Total_cases,sum(TotalDeaths) as Total_deaths,sum(TotalRecovered) as total_recovered,
        sum(ActiveCases) as total_activecases,sum(Population) as total_population
from df6
group by Continent
order by Total_cases desc
"""
result = spark.sql(trend_by_continent)
result.show()

+-----------------+-----------+------------+---------------+-----------------+----------------+
|        Continent|Total_cases|Total_deaths|total_recovered|total_activecases|total_population|
+-----------------+-----------+------------+---------------+-----------------+----------------+
|    North America|    5919209|      229855|        3151678|          2537676|       589503467|
|             Asia|    4689794|      100627|        3508170|          1080997|      3173656415|
|    South America|    4543273|      154885|        3116150|          1272238|       431110464|
|           Europe|    2982576|      205232|        1587302|           475261|       747677546|
|           Africa|    1011867|       22114|         693620|           296133|      1343515489|
|Australia/Oceania|      21735|         281|          12620|             8834|        40957909|
|             NULL|        712|          13|            651|               48|            NULL|
+-----------------+-----------+---------

### Q14.Continent That Has The Lowest Cases

In [87]:
lowest_cases = """ 
select Continent,min(TotalCases) as lowest_case
from df6
group by Continent
"""
result = spark.sql(lowest_cases )
result.show()

+-----------------+-----------+
|        Continent|lowest_case|
+-----------------+-----------+
|           Europe|         12|
|           Africa|         10|
|             NULL|        712|
|Australia/Oceania|         22|
|    North America|         13|
|    South America|         13|
|             Asia|         20|
+-----------------+-----------+

